In [1]:
"""
Data cleaning

Turns the raw databento ES continuous-futures export into a clean OHLCV
parquet in data/processed/. This is the only notebook that writes to
data/processed/ — everything else (data_exploration, etc.) only reads from it.
"""

import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")
RAW_FILE = DATA_DIR / "raw" / "ES_1m_2010_2026_full.parquet"
PROCESSED_FILE = DATA_DIR / "processed" / "es_1min_bars_2010_2026.parquet"

pd.set_option("display.width", 120)

## Load raw + clean

In [2]:
raw = pd.read_parquet(RAW_FILE)
raw.head()

,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol
ts_event,,,,,,,,,
2010-07-01 00:00:00+00:00,33,1,26714,1026.25,1026.25,1026.00,1026.00,225,ES.c.0
2010-07-01 00:01:00+00:00,33,1,26714,1026.25,1026.25,1026.00,1026.25,90,ES.c.0
2010-07-01 00:02:00+00:00,33,1,26714,1026.25,1026.25,1026.00,1026.25,86,ES.c.0
2010-07-01 00:03:00+00:00,33,1,26714,1026.00,1026.50,1026.00,1026.50,111,ES.c.0
2010-07-01 00:04:00+00:00,33,1,26714,1026.25,1027.00,1026.25,1027.00,781,ES.c.0


In [3]:
# Drop databento request/session metadata that isn't part of the price series
# (rtype/publisher_id are constant; instrument_id changes as the continuous
# contract rolls and isn't needed for price analysis).
df = raw[["open", "high", "low", "close", "volume"]].copy()
df.index.name = "timestamp"
df = df.sort_index()
df.head()

,open,high,low,close,volume
timestamp,,,,,
2010-07-01 00:00:00+00:00,1026.25,1026.25,1026.00,1026.00,225
2010-07-01 00:01:00+00:00,1026.25,1026.25,1026.00,1026.25,90
2010-07-01 00:02:00+00:00,1026.25,1026.25,1026.00,1026.25,86
2010-07-01 00:03:00+00:00,1026.00,1026.50,1026.00,1026.50,111
2010-07-01 00:04:00+00:00,1026.25,1027.00,1026.25,1027.00,781


In [4]:
PROCESSED_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(PROCESSED_FILE)
print(f"wrote {len(df):,} rows -> {PROCESSED_FILE}")

wrote 4,811,336 rows -> ../data/processed/es_1min_bars_2010_2026.parquet


**Findings:** Dropped databento's `rtype`/`publisher_id`/`instrument_id` columns
(session metadata, not price data) and wrote 4,811,336 clean OHLCV rows to
`data/processed/es_1min_bars_2010_2026.parquet`.